# ConvTran Reproduction: UEA Baseline Training

We evaluate the released ConvTran implementation on the 30 datasets in the UEA multivariate archive. The BasicMotions pilot is documented in `01_reproduction.ipynb`; archive preparation and dataset dimensions are documented in `02_uea_benchmark.ipynb`. This notebook records the baseline training protocol, completed results, and limitations encountered as the benchmark progresses.

- Paper: https://doi.org/10.1007/s10618-023-00948-2
- Official code: https://github.com/Navidfoumani/ConvTran
- Project repository: https://github.com/tyrerodr/convtran-reproduction

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

ROOT = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "upstream" / "ConvTran" / "main.py").is_file()
    ),
    None,
)
assert ROOT is not None, "Open this notebook from the project repository."

DATA_ROOT = ROOT / "Dataset" / "UEA" / "Multivariate_ts"
RUNNER = ROOT / "scripts" / "run_baseline.py"
RUNS_PATH = ROOT / "results" / "baseline_runs.csv"
DIMENSIONS_PATH = ROOT / "results" / "dataset_dimensions.csv"

assert RUNNER.is_file(), f"Missing baseline runner: {RUNNER}"
print("Repository:", ROOT)
print("Python:", sys.executable)
print("Upstream commit:", subprocess.check_output(
    ["git", "-C", str(ROOT / "upstream" / "ConvTran"), "rev-parse", "HEAD"],
    text=True,
).strip())

Repository: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction
Python: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/bin/python
Upstream commit: 148afb6ca549915b7e78b05e2ec3b4ba6e052341


## Baseline protocol

We keep the released architecture and default training configuration used in the BasicMotions pilot. We call this the *released-code baseline* because the paper's method description differs from some implementation defaults.

| Setting | Baseline value |
|---|---|
| Positional encoding | tAPE and eRPE |
| Embedding dimension | 16 |
| Attention heads | 8 |
| Feedforward dimension | 256 |
| Optimizer | RAdam |
| Learning rate | 0.001 |
| Dropout | 0.01 |
| Epochs | 100 |
| Batch size | 16 |
| Normalization | Disabled |
| Training seeds | 1234, 2345, 3456, 4567, 5678 |
| Model selection | Lowest validation loss |

Each seed runs in a separate Python process. The upstream loader creates one stratified validation split with a fixed random state of 1234, independent of the training seed. Thus the five runs measure variation in training, not variation across data splits. Test accuracy is computed on the official UEA test split after checkpoint selection.

## Dataset scope

The dataset inventory contains 30 TRAIN/TEST pairs. It records sample counts, channel counts, class counts, and sequence lengths without loading all series into memory. File presence and the inventory do not guarantee that every dataset can be trained with the available GPU memory. In particular, EigenWorms has 17,984 time steps and is scheduled last; its feasibility with the unchanged baseline remains unverified.

In [2]:
UEA_DATASETS = [
    "ArticularyWordRecognition", "AtrialFibrillation", "BasicMotions",
    "CharacterTrajectories", "Cricket", "DuckDuckGeese", "EigenWorms",
    "Epilepsy", "ERing", "EthanolConcentration", "FaceDetection",
    "FingerMovements", "HandMovementDirection", "Handwriting", "Heartbeat",
    "InsectWingbeat", "JapaneseVowels", "Libras", "LSST", "MotorImagery",
    "NATOPS", "PenDigits", "PEMS-SF", "PhonemeSpectra", "RacketSports",
    "SelfRegulationSCP1", "SelfRegulationSCP2", "SpokenArabicDigits",
    "StandWalkJump", "UWaveGestureLibrary",
]
EXPECTED_SEEDS = {1234, 2345, 3456, 4567, 5678}
assert len(UEA_DATASETS) == len(set(UEA_DATASETS)) == 30

available = {
    name for name in UEA_DATASETS
    if all(
        (DATA_ROOT / name / f"{name}_{split}.ts").is_file()
        for split in ("TRAIN", "TEST")
    )
}
print(f"Datasets with both split files: {len(available)}/30")
if missing := sorted(set(UEA_DATASETS) - available):
    raise FileNotFoundError(f"Missing split files: {missing}")

if DIMENSIONS_PATH.is_file():
    dimensions = pd.read_csv(DIMENSIONS_PATH)
    display(dimensions[
        ["dataset", "train_samples", "test_samples",
         "channels", "classes", "max_length", "variable_length"]
    ])
else:
    print("Dataset dimensions CSV not found; see notebook 02.")

Datasets with both split files: 30/30


,dataset,train_samples,test_samples,channels,classes,max_length,variable_length
0,ArticularyWordRecognition,275,300,9,25,144,False
1,AtrialFibrillation,15,15,2,3,640,False
2,BasicMotions,40,40,6,4,100,False
3,CharacterTrajectories,1422,1436,3,20,182,True
4,Cricket,108,72,6,12,1197,False
5,DuckDuckGeese,50,50,1345,5,270,False
6,EigenWorms,128,131,6,5,17984,False
7,Epilepsy,137,138,3,4,206,False
8,ERing,30,270,4,6,65,False
9,EthanolConcentration,261,263,3,4,1751,False


## Training

The runner invokes the official `main.py` separately for each seed and writes logs and checkpoints under `outputs/baseline/<dataset>/seed<seed>/`. It reuses a completed run when its log contains the final test summary. It writes a dataset's rows to `results/baseline_runs.csv` after all five seeds have completed. An incomplete log must be inspected before that seed is retried.

The cell below trains datasets that are not yet complete in the CSV. It runs one dataset at a time; if one fails, the failure is reported and the next dataset is attempted. No model or batch-size change is made automatically. Running this cell starts training and may take substantial time.

In [3]:
def load_runs():
    if not RUNS_PATH.is_file():
        return pd.DataFrame(columns=[
            "dataset", "seed", "epochs", "batch_size", "accuracy", "test_loss"
        ])
    runs = pd.read_csv(RUNS_PATH)
    expected = [
        "dataset", "seed", "epochs", "batch_size", "accuracy", "test_loss"
    ]
    if list(runs.columns) != expected:
        raise ValueError(f"Unexpected results columns: {list(runs.columns)}")
    if runs.duplicated(["dataset", "seed"]).any():
        raise ValueError("Duplicate dataset/seed rows in the results CSV")
    return runs

def completed_datasets(runs):
    return {
        dataset for dataset, group in runs.groupby("dataset")
        if len(group) == 5
        and set(group["seed"].astype(int)) == EXPECTED_SEEDS
    }

runs = load_runs()
print("Completed:", sorted(completed_datasets(runs)))
print("Pending:", sorted(set(UEA_DATASETS) - completed_datasets(runs)))

Completed: ['ArticularyWordRecognition', 'AtrialFibrillation', 'BasicMotions', 'CharacterTrajectories', 'Cricket', 'DuckDuckGeese', 'ERing', 'Epilepsy', 'EthanolConcentration', 'FaceDetection', 'FingerMovements', 'HandMovementDirection', 'Handwriting', 'Heartbeat', 'InsectWingbeat', 'JapaneseVowels', 'LSST', 'Libras', 'NATOPS', 'PEMS-SF', 'PenDigits']
Pending: ['EigenWorms', 'MotorImagery', 'PhonemeSpectra', 'RacketSports', 'SelfRegulationSCP1', 'SelfRegulationSCP2', 'SpokenArabicDigits', 'StandWalkJump', 'UWaveGestureLibrary']


In [4]:
training_order = [
    name for name in UEA_DATASETS if name != "EigenWorms"
] + ["EigenWorms"]

attempts = []
for dataset in training_order:
    if dataset in completed_datasets(load_runs()):
        continue

    print(f"\nTraining {dataset}", flush=True)
    process = subprocess.run(
        [sys.executable, str(RUNNER), "--dataset", dataset],
        cwd=ROOT,
        check=False,
    )
    attempts.append({
        "dataset": dataset,
        "exit_code": process.returncode,
    })
    if process.returncode:
        print(
            f"{dataset} failed. Inspect "
            f"{ROOT / 'outputs' / 'baseline' / dataset.lower()}",
            flush=True,
        )

display(pd.DataFrame(attempts, columns=["dataset", "exit_code"]))


Training MotorImagery
MotorImagery failed. Inspect /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/baseline/motorimagery

Training PhonemeSpectra
PhonemeSpectra failed. Inspect /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/baseline/phonemespectra

Training RacketSports
Training RacketSports, seed=1234. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/baseline/racketsports/seed1234/run.log


Incomplete log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/baseline/motorimagery/seed1234/run.log. Inspect and rename it before retrying.
Incomplete log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/baseline/phonemespectra/seed1234/run.log. Inspect and rename it before retrying.


Training RacketSports, seed=2345. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/baseline/racketsports/seed2345/run.log
Training RacketSports, seed=3456. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/baseline/racketsports/seed3456/run.log
Training RacketSports, seed=4567. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/baseline/racketsports/seed4567/run.log
Training RacketSports, seed=5678. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/baseline/racketsports/seed5678/run.log
Saved results to: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/results/baseline_runs.csv

Training SelfRegulationSCP1
Training SelfRegulationSCP1, seed=1234. Log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/baseline/selfregulationscp1/seed1234/run.log
Training SelfRegulationSCP1, seed=2345. L

Incomplete log: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/baseline/eigenworms/seed1234/run.log. Inspect and rename it before retrying.


,dataset,exit_code
0,MotorImagery,1
1,PhonemeSpectra,1
2,RacketSports,0
3,SelfRegulationSCP1,0
4,SelfRegulationSCP2,0
5,SpokenArabicDigits,0
6,EigenWorms,1


## Results collected so far

The released-code baseline completed five runs on 28 of the 30 UEA datasets with 100 epochs and batch size 16. MotorImagery and EigenWorms did not complete because the attention operation ran out of GPU memory at the first epoch. Their failed logs remain under `outputs/baseline/`; neither dataset has a baseline result in the CSV.

The table below reads `results/baseline_runs.csv` and reports mean test accuracy and sample standard deviation across the five training seeds. It is the source for numerical claims in this notebook. The five runs share one validation split, and equal accuracies do not imply identical predictions.

In [5]:
runs = load_runs()
completed = completed_datasets(runs)

if runs.empty:
    print("No baseline results have been recorded.")
else:
    summary = (
        runs.assign(accuracy_pct=runs["accuracy"] * 100)
        .groupby("dataset")
        .agg(
            runs=("seed", "count"),
            mean_accuracy_pct=("accuracy_pct", "mean"),
            std_accuracy_pp=("accuracy_pct", "std"),
        )
        .reset_index()
    )
    summary = summary[summary["dataset"].isin(completed)]
    display(summary.round(4).sort_values("dataset"))

progress = pd.DataFrame({"dataset": UEA_DATASETS})
progress["complete"] = progress["dataset"].isin(completed)
print(f"Completed datasets: {len(completed)}/30")
display(progress)

,dataset,runs,mean_accuracy_pct,std_accuracy_pp
0,ArticularyWordRecognition,5,96.8000,0.8028
1,AtrialFibrillation,5,32.0000,2.9814
2,BasicMotions,5,100.0000,0.0000
3,CharacterTrajectories,5,99.1643,0.3229
4,Cricket,5,98.3333,1.1620
5,DuckDuckGeese,5,61.2000,1.0954
6,ERing,5,84.1481,3.8792
7,Epilepsy,5,96.3768,0.0000
8,EthanolConcentration,5,29.2015,1.1215
9,FaceDetection,5,65.4200,1.5402


Completed datasets: 28/30


,dataset,complete
0,ArticularyWordRecognition,True
1,AtrialFibrillation,True
2,BasicMotions,True
3,CharacterTrajectories,True
4,Cricket,True
5,DuckDuckGeese,True
6,EigenWorms,False
7,Epilepsy,True
8,ERing,True
9,EthanolConcentration,True


## Reproducibility findings and limitations

The released code differs from the paper's description in embedding dimension (16 by default versus 64 in the paper), optimizer (RAdam versus Adam), feedforward activation (ReLU versus GELU), and training termination (fixed epoch budget with a best validation-loss checkpoint versus described early stopping). Changing only the embedding dimension also changes the temporal convolution filter count. We therefore label these results as a released-code baseline.

The upstream loader fixes the validation split independently of the training seed and does not invalidate its processed-data cache when preprocessing options change. These details limit the interpretation of variability and require care in later comparisons.

MotorImagery has 3,000 time steps; at batch size 16 its attention matrix attempted an additional 4.29 GiB allocation. EigenWorms has 17,984 time steps and attempted 154.22 GiB. These values come from the failed seed-1234 logs on our 12 GB GPU. A smaller batch or reduced sequence length would be a separately recorded adaptation, not a completed run of this baseline.

## Comparison with the published results

Table 2 of the ConvTran paper reports average accuracy across datasets. Our five-seed means are preliminary comparisons because the released-code defaults differ from the paper's description and the complete published training protocol is not available. The largest gaps in our completed runs are Handwriting (23.58% versus 37.52%), ERing (84.15% versus 96.29%), PEMS-SF (72.83% versus 82.84%), and Libras (83.78% versus 92.77%). BasicMotions matches the paper at 100%. These differences are descriptive; we have not established their causes.

Paper: https://doi.org/10.1007/s10618-023-00948-2

MotorImagery and EigenWorms remain incomplete under the unchanged baseline. Any result obtained with a smaller batch or altered sequence length will be reported separately. The next notebook will define and evaluate an experimental improvement after the baseline discrepancies have been reviewed.
